In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/shashwatkumarjha/thodisibadiclip/Thodisibadiclip.mp4
/kaggle/input/datasets/shashwatkumarjha/wajjan/best.pt


In [2]:
%pip -q install ultralytics openpyxl scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 5.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
os.environ["MPLBACKEND"] = "Agg"

import cv2
import numpy as np
import pandas as pd
import pathlib
from collections import defaultdict, deque
import itertools
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.ndimage import median_filter
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

from ultralytics import YOLO

print("✅ All dependencies loaded successfully! Kalman filter and ByteTrack ready.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
✅ All dependencies loaded successfully! Kalman filter and ByteTrack ready.


In [5]:
# ================================================================
# 1. FILE & VIDEO PATHS (Supports Kaggle or Local folders)
# ================================================================
MODEL_CANDIDATES = [
    "/kaggle/input/datasets/shashwatkumarjha/wajjan/best.pt",
    "best.pt",
    "yolov8n.pt"  # Fallback pre-trained model if custom weights are absent
]
VIDEO_CANDIDATES = [
    "/kaggle/input/datasets/shashwatkumarjha/thodisibadiclip/Thodisibadiclip.mp4",
    "final_output.mp4",
    "Thodisibadiclip.mp4"
]

def resolve_existing_path(paths, default_val):
    for p in paths:
        if pathlib.Path(p).exists():
            return p
    return default_val

MODEL_PATH     = resolve_existing_path(MODEL_CANDIDATES, MODEL_CANDIDATES[0])
INPUT_VIDEO    = resolve_existing_path(VIDEO_CANDIDATES, VIDEO_CANDIDATES[0])

# Video speed modification option
# Set ENABLE_VIDEO_SLOWDOWN = True to generate a 2x slowed video before tracking
ENABLE_VIDEO_SLOWDOWN = False
SLOWDOWN_FACTOR       = 0.5   # 0.5 = 50% speed (2x slower, doubles frame count)
SLOW_VIDEO_PATH       = "final_output_slow.mp4"

# Downstream output destinations
VIDEO_OUT_PATH = "final_output_tracked.mp4"
XLSX_OUT_PATH  = "collision_log.xlsx"
HM_MTTC_PATH   = "heatmap_mttc_rearend.png"
HM_PET_PATH    = "heatmap_pet_crossing.png"
HM_COMBO_PATH  = "heatmap_combined.png"

# ================================================================
# 2. TRACKING & KINEMATICS HYPERPARAMETERS
# ================================================================
DETECTION_CONFIDENCE   = 0.45   # Increased from 0.35 to eliminate noisy false detections
ALPHA_VEL              = 0.15   # Velocity EMA smoothing
ALPHA_ACC              = 0.05   # Acceleration EMA smoothing
ALPHA_SPEED            = 0.15   # Display speed smoothing
TRAJ_FIT_MIN_PTS       = 6      # Minimum points for trajectory regression
TRAJ_FIT_WINDOW        = 20     # Window size for regression velocity
CLEANUP_EVERY          = 100    # Purge stale tracks every N frames

# 3. KALMAN FILTER & RE-ID CONTINUITY
MAX_COAST_FRAMES       = 3      # Keep predicting lost tracks for up to 3 frames
REID_MAX_DISTANCE_M    = 2.2    # Distance threshold (m) to re-associate fragmented tracks
KF_PROCESS_VAR_POS     = 0.04   # Kalman process variance for position
KF_PROCESS_VAR_VEL     = 0.15   # Kalman process variance for velocity
KF_MEAS_VAR            = 0.12   # Kalman measurement noise variance

# 4. CONFLICT DETECTION GATES
DISTANCE_GATE_M        = 40.0   # Skip pairs farther than 40m
MIN_APPROACH_DIST_M    = 1.5    # Closest approach threshold for conflict
SIDE_BY_SIDE_LATERAL_M = 1.0    # Lateral separation guard for parallel lanes
TTC_MAX_S              = 5.0    # Ignore TTC beyond 5 seconds

FOLLOWING_ANGLE_DEG    = 25.0   # Angular cone for rear-end scenario
FOLLOWING_LONGIT_GAP_M = 1.5

# 5. REFINED RISK PERSISTENCE (Window-based)
RISK_PERSIST_FRAMES    = 5      # Number of hits required to trigger alert
RISK_WINDOW_SIZE       = 7      # Window of recent frames (allows 1-2 missed frames)

# Safety thresholds
DRAC_CRITICAL          = 3.4    # m/s² (AASHTO standard)
DRAC_WARNING           = 1.0
MTTC_CRITICAL_S        = 1.5    # seconds
MTTC_WARNING_S         = 3.0
PET_CRITICAL_S         = 1.0    # seconds
PET_WARNING_S          = 2.0

# 6. ROAD BOUNDS & FINER HEATMAP GRID
WORLD_X_MIN, WORLD_X_MAX = 0.0, 3.0
WORLD_Y_MIN, WORLD_Y_MAX = 0.0, 74.65
GRID_X_STEP = 1.0     # 1 m across road
GRID_Y_STEP = 5.0     # Finer 5 m along road (was 10.0 m) to capture localized conflict clusters
LOG_INTERVAL_FRAMES = 15

# 7. VISUAL STYLES & HUD SCALING
HUD_FONT_SCALE       = 0.65    # Enlarged from 0.42 for projector/high-res readability
HUD_ROW_HEIGHT       = 28      # Height per HUD alert line
VEHICLE_LABEL_SCALE  = 0.50    # Crisp vehicle tag font
HEATMAP_LABEL_FONT   = 11      # Larger font for heatmap annotations

COLOR_TRAJ_PATH   = (0, 140, 255)
COLOR_GRID        = (50, 50, 50)
COLOR_GRID_LABEL  = (130, 130, 60)
COLOR_AXIS_X      = (180, 60, 60)
COLOR_AXIS_Y      = (60, 180, 60)

class_names  = {0: "Car", 1: "Person", 2: "2-Wheeler"}
class_colors = {0: (0, 255, 0), 1: (255, 128, 0), 2: (0, 200, 255)}

print(f"📌 Model path : {MODEL_PATH}")
print(f"📌 Input video: {INPUT_VIDEO}")

📌 Model path : /kaggle/input/datasets/shashwatkumarjha/wajjan/best.pt
📌 Input video: /kaggle/input/datasets/shashwatkumarjha/thodisibadiclip/Thodisibadiclip.mp4


In [6]:
scale_x = 960 / 1920
scale_y = 540 / 1080

img_pts = np.array([
    [972  * scale_x, 216  * scale_y],
    [1080 * scale_x, 216  * scale_y],
    [1650 * scale_x, 1066 * scale_y],
    [540  * scale_x, 1066 * scale_y],
], dtype=np.float32)

world_pts = np.array([
    [0.0, 74.65],
    [3.0, 74.65],
    [3.0, 0.0],
    [0.0, 0.0],
], dtype=np.float32)

H, _ = cv2.findHomography(img_pts, world_pts)
H_inv = np.linalg.inv(H)

# Pre-calculate ROI polygon for coordinate boundary tests
roi_pts = np.array([
    [int(972  * scale_x), int(216  * scale_y)],
    [int(1080 * scale_x), int(216  * scale_y)],
    [int(1650 * scale_x), int(1066 * scale_y)],
    [int(540  * scale_x), int(1066 * scale_y)],
], dtype=np.int32)

def pixel_to_world(u, v):
    pt = H @ np.array([u, v, 1.0])
    pt /= pt[2]
    return float(pt[0]), float(pt[1])

def world_to_pixel(X, Y):
    pt = H_inv @ np.array([X, Y, 1.0])
    pt /= pt[2]
    return int(pt[0]), int(pt[1])

# Pre-compute world grid lines & labels
_grid_lines = []
_grid_labels = []

def _build_grid():
    for x in np.arange(WORLD_X_MIN, WORLD_X_MAX + GRID_X_STEP * 0.5, GRID_X_STEP):
        p0 = world_to_pixel(x, WORLD_Y_MIN)
        p1 = world_to_pixel(x, WORLD_Y_MAX)
        is_axis = abs(x - WORLD_X_MIN) < 1e-4
        _grid_lines.append((p0, p1, COLOR_AXIS_Y if is_axis else COLOR_GRID, 2 if is_axis else 1))
        lp = world_to_pixel(x, WORLD_Y_MIN + 3.0)
        _grid_labels.append((lp[0] + 3, lp[1], f"x={x:.0f}m"))

    for y in np.arange(WORLD_Y_MIN, WORLD_Y_MAX + GRID_Y_STEP * 0.5, GRID_Y_STEP):
        p0 = world_to_pixel(WORLD_X_MIN, y)
        p1 = world_to_pixel(WORLD_X_MAX, y)
        is_axis = abs(y - WORLD_Y_MIN) < 1e-4
        _grid_lines.append((p0, p1, COLOR_AXIS_X if is_axis else COLOR_GRID, 2 if is_axis else 1))
        lp = world_to_pixel(WORLD_X_MIN, y)
        _grid_labels.append((lp[0] + 4, lp[1] - 3, f"{y:.0f}m"))

_build_grid()

def draw_world_grid(frame):
    ov = frame.copy()
    for (p0, p1, color, thick) in _grid_lines:
        cv2.line(ov, p0, p1, color, thick, cv2.LINE_AA)
    cv2.addWeighted(ov, 0.55, frame, 0.45, 0, frame)
    for (lx, ly, txt) in _grid_labels:
        (tw, th), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.32, 1)
        cv2.rectangle(frame, (lx-1, ly-th-1), (lx+tw+1, ly+2), (10, 10, 10), -1)
        cv2.putText(frame, txt, (lx, ly), cv2.FONT_HERSHEY_SIMPLEX, 0.32, COLOR_GRID_LABEL, 1, cv2.LINE_AA)
    cv2.putText(frame, "-- x-axis (road width, m)", (650, 508), cv2.FONT_HERSHEY_SIMPLEX, 0.36, COLOR_AXIS_Y, 1)
    cv2.putText(frame, "-- y-axis (road length, m)", (650, 524), cv2.FONT_HERSHEY_SIMPLEX, 0.36, COLOR_AXIS_X, 1)

In [7]:
# Cell 5: 2D Centroid Kalman Filter & Re-ID Continuity
class CentroidKalmanFilter:
    """
    Constant-Velocity Kalman Filter for World Coordinates [X, Y, Vx, Vy].
    Suppresses perspective homography jitter for distant vehicles.
    """
    def __init__(self, init_pos, dt=1.0/30.0):
        self.dt = dt
        # State: [x, y, vx, vy]
        self.x = np.array([init_pos[0], init_pos[1], 0.0, 0.0], dtype=np.float64)
        
        self.F = np.array([
            [1.0, 0.0, dt,  0.0],
            [0.0, 1.0, 0.0, dt ],
            [0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, 1.0]
        ], dtype=np.float64)
        
        self.H = np.array([
            [1.0, 0.0, 0.0, 0.0],
            [0.0, 1.0, 0.0, 0.0]
        ], dtype=np.float64)
        
        self.P = np.diag([0.5, 0.5, 2.0, 2.0]).astype(np.float64)
        
        qp = KF_PROCESS_VAR_POS
        qv = KF_PROCESS_VAR_VEL
        self.Q = np.diag([qp, qp, qv, qv]).astype(np.float64)
        
        r_val = KF_MEAS_VAR
        self.R = np.diag([r_val, r_val]).astype(np.float64)

    def predict(self, dt=None):
        if dt is not None and abs(dt - self.dt) > 1e-4:
            self.dt = dt
            self.F[0, 2] = dt
            self.F[1, 3] = dt
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        return self.x[:2].copy()

    def update(self, meas):
        z = np.array(meas, dtype=np.float64)
        y = z - (self.H @ self.x)
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        I = np.eye(4, dtype=np.float64)
        self.P = (I - K @ self.H) @ self.P
        return self.x[:2].copy()

    def get_pos(self):
        return self.x[:2].copy()

    def get_vel(self):
        return self.x[2:].copy()

class ReIDBridgeTracker:
    """
    Coasts tracks over short detection gaps and maps new fragmented ByteTrack IDs
    back to the consistent original track ID to eliminate ID-flips.
    """
    def __init__(self, max_coast_frames=MAX_COAST_FRAMES, max_reid_dist=REID_MAX_DISTANCE_M):
        self.max_coast_frames = max_coast_frames
        self.max_reid_dist = max_reid_dist
        self.filters = {}         # canonical_id -> CentroidKalmanFilter
        self.coast_counts = {}    # canonical_id -> missed frame count
        self.id_aliases = {}      # current_bytetrack_id -> canonical_id
        self.last_classes = {}    # canonical_id -> class label
        self.last_bboxes = {}     # canonical_id -> bbox

    def step(self, raw_detections, dt):
        """
        raw_detections: list of dicts {'id', 'cls', 'world_pos', 'bbox', 'px'}
        Returns: list of dicts with 'canonical_id' and smoothed 'world_pos'
        """
        # Predict all existing Kalman filters
        for cid, kf in self.filters.items():
            kf.predict(dt)
            self.coast_counts[cid] += 1

        matched_cids = set()
        output_tracks = []

        # 1. Match detections that already have an alias or direct ID
        unmatched_dets = []
        for det in raw_detections:
            raw_id = det['id']
            cid = self.id_aliases.get(raw_id, raw_id)

            if cid in self.filters and self.coast_counts[cid] <= self.max_coast_frames:
                # Direct hit
                smoothed_pos = self.filters[cid].update(det['world_pos'])
                self.coast_counts[cid] = 0
                matched_cids.add(cid)
                self.last_classes[cid] = det['cls']
                self.last_bboxes[cid] = det['bbox']
                det_out = det.copy()
                det_out['id'] = cid
                det_out['world_pos'] = smoothed_pos
                output_tracks.append(det_out)
            else:
                unmatched_dets.append(det)

        # 2. Re-ID association: For new IDs, check if they match a recently coasting track
        coasting_cids = [cid for cid in self.filters if cid not in matched_cids and self.coast_counts[cid] <= self.max_coast_frames]

        for det in unmatched_dets:
            raw_id = det['id']
            det_pos = det['world_pos']
            best_cid = None
            best_dist = self.max_reid_dist

            for cid in coasting_cids:
                if cid in matched_cids:
                    continue
                if self.last_classes.get(cid) == det['cls']:
                    pred_pos = self.filters[cid].get_pos()
                    dist = np.linalg.norm(pred_pos - det_pos)
                    if dist < best_dist:
                        best_dist = dist
                        best_cid = cid

            if best_cid is not None:
                # Re-associated! Map raw_id to canonical ID
                self.id_aliases[raw_id] = best_cid
                smoothed_pos = self.filters[best_cid].update(det_pos)
                self.coast_counts[best_cid] = 0
                matched_cids.add(best_cid)
                self.last_classes[best_cid] = det['cls']
                self.last_bboxes[best_cid] = det['bbox']
                det_out = det.copy()
                det_out['id'] = best_cid
                det_out['world_pos'] = smoothed_pos
                output_tracks.append(det_out)
            else:
                # Brand new track
                cid = raw_id
                self.id_aliases[raw_id] = cid
                self.filters[cid] = CentroidKalmanFilter(det_pos, dt)
                self.coast_counts[cid] = 0
                matched_cids.add(cid)
                self.last_classes[cid] = det['cls']
                self.last_bboxes[cid] = det['bbox']
                det_out = det.copy()
                det_out['id'] = cid
                output_tracks.append(det_out)

        # Purge dead tracks beyond coast limit
        dead_cids = [cid for cid, c in self.coast_counts.items() if c > self.max_coast_frames * 2]
        for cid in dead_cids:
            self.filters.pop(cid, None)
            self.coast_counts.pop(cid, None)
            self.last_classes.pop(cid, None)
            self.last_bboxes.pop(cid, None)

        return output_tracks

In [8]:
# Cell 6: Kinematics & Adaptive Motion Estimation
NOMINAL_FPS = 30.0
dt_nominal = 1.0 / NOMINAL_FPS

def adaptive_alpha(alpha, dt, nom_dt=dt_nominal):
    return 1.0 - (1.0 - alpha) ** max(0.1, min(5.0, dt / nom_dt))

track_history_world = defaultdict(lambda: deque(maxlen=40))
track_history_px    = defaultdict(lambda: deque(maxlen=40))
track_speed         = defaultdict(float)
velocity_ema        = defaultdict(lambda: np.zeros(2))
accel_ema           = defaultdict(lambda: np.zeros(2))
prev_vel_for_acc    = defaultdict(lambda: np.zeros(2))
track_id_to_class   = {}

# Refined risk persistence: deque of booleans per pair
pair_risk_window    = defaultdict(lambda: deque(maxlen=RISK_WINDOW_SIZE))
stable_alerts       = {}

def compute_ema_velocity(tid, dt):
    hist = track_history_world[tid]
    if len(hist) < 2:
        return np.zeros(2)
    raw = (hist[-1] - hist[-2]) / dt
    a = adaptive_alpha(ALPHA_VEL, dt)
    v = a * raw + (1.0 - a) * velocity_ema[tid]
    velocity_ema[tid] = v
    return v

def fit_regression_velocity(tid):
    hist = list(track_history_world[tid])
    n = len(hist)
    if n < TRAJ_FIT_MIN_PTS:
        return velocity_ema[tid].copy()
    pts = np.array(hist[-TRAJ_FIT_WINDOW:])
    t = np.arange(len(pts), dtype=float) * dt_nominal
    vx = np.polyfit(t, pts[:, 0], 1)[0]
    vy = np.polyfit(t, pts[:, 1], 1)[0]
    return np.array([vx, vy])

def compute_acceleration(tid, vel, dt):
    raw = (vel - prev_vel_for_acc[tid]) / dt
    a = adaptive_alpha(ALPHA_ACC, dt)
    acc = a * raw + (1.0 - a) * accel_ema[tid]
    accel_ema[tid] = acc
    prev_vel_for_acc[tid] = vel
    return acc

In [9]:
# Cell 7: Direction-Aware Conflict Metrics (DRAC, MTTC, PET)
def closest_approach(p1, v1, p2, v2):
    rel_p = p2 - p1
    rel_v = v2 - v1
    denom = np.dot(rel_v, rel_v)
    if denom < 1e-6:
        return 0.0, float(np.linalg.norm(rel_p)), (p1 + p2) / 2.0
    t_ca = -np.dot(rel_p, rel_v) / denom
    fp1 = p1 + v1 * t_ca
    fp2 = p2 + v2 * t_ca
    return t_ca, float(np.linalg.norm(fp1 - fp2)), (fp1 + fp2) / 2.0

def lateral_separation(p1, v1, p2):
    s = np.linalg.norm(v1)
    if s < 1e-3:
        return float(np.linalg.norm(p2 - p1))
    perp = np.array([-v1[1], v1[0]]) / s
    return abs(np.dot(p2 - p1, perp))

def is_following_scenario(v1, v2, p1, p2):
    s1, s2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if s1 < 0.5 or s2 < 0.5:
        return False
    if np.dot(v1/s1, v2/s2) < np.cos(np.radians(FOLLOWING_ANGLE_DEG)):
        return False
    avg = (v1/s1 + v2/s2)
    avg /= (np.linalg.norm(avg) + 1e-9)
    return abs(np.dot(p2 - p1, avg)) > 0.1

def compute_drac(p1, p2, v1, v2):
    """
    Direction-Aware DRAC: Projects closing speed along line of sight.
    Diverging vehicles give DRAC = 0 (prevents false alerts).
    """
    rel_p = p2 - p1
    S = np.linalg.norm(rel_p)
    if S < 1e-3:
        return 0.0
    los = rel_p / S
    closing_speed = np.dot(v1 - v2, los)
    if closing_speed <= 0.1:
        return 0.0
    return float((closing_speed ** 2) / S)

def compute_mttc(p1, v1, a1, p2, v2, a2):
    rel_p = p2 - p1
    dist = np.linalg.norm(rel_p)
    if dist < 1e-3:
        return np.inf
    d = rel_p / dist
    S = np.dot(rel_p, d)
    dv = np.dot(v2 - v1, d)
    da = np.dot(a2 - a1, d)
    if abs(da) < 1e-5:
        return (-S / dv) if dv < 0 else np.inf
    disc = dv**2 + 2.0 * da * S
    if disc < 0:
        return np.inf
    sq = np.sqrt(disc)
    valid = [t for t in [(-dv + sq) / da, (-dv - sq) / da] if t > 0]
    return float(min(valid)) if valid else np.inf

def compute_pet(p1, v1, p2, v2, p_ca):
    def time_to_point(p, v, target):
        speed = np.linalg.norm(v)
        if speed < 0.5:
            return np.inf
        v_hat = v / speed
        proj = np.dot(target - p, v_hat)
        if proj < 0:
            return np.inf
        return proj / speed

    t1 = time_to_point(p1, v1, p_ca)
    t2 = time_to_point(p2, v2, p_ca)
    if t1 == np.inf or t2 == np.inf:
        return np.inf
    return abs(t1 - t2)

def risk_tier_rear_end(TTC, DRAC_val, MTTC_val):
    if (TTC < MTTC_CRITICAL_S and DRAC_val >= DRAC_CRITICAL and MTTC_val < MTTC_CRITICAL_S):
        return "CRITICAL", (0, 0, 255)
    if (TTC < MTTC_WARNING_S or DRAC_val >= DRAC_WARNING or MTTC_val < MTTC_WARNING_S):
        return "WARNING", (0, 140, 255)
    return "MONITOR", (0, 210, 210)

def risk_tier_crossing(PET_val):
    if PET_val < PET_CRITICAL_S:
        return "CRITICAL", (0, 0, 255)
    if PET_val < PET_WARNING_S:
        return "WARNING", (0, 140, 255)
    return "MONITOR", (0, 210, 210)

In [10]:
# Cell 8: Finer Heatmap Grid & High-Visibility HUD
_hm_nx = max(1, int(np.ceil((WORLD_X_MAX - WORLD_X_MIN) / GRID_X_STEP)))
_hm_ny = max(1, int(np.ceil((WORLD_Y_MAX - WORLD_Y_MIN) / GRID_Y_STEP)))

heatmap_mttc = np.zeros((_hm_ny, _hm_nx), dtype=np.float32)
heatmap_pet  = np.zeros((_hm_ny, _hm_nx), dtype=np.float32)

def world_to_heatmap_cell(X, Y):
    col = int((X - WORLD_X_MIN) / GRID_X_STEP)
    row = int((Y - WORLD_Y_MIN) / GRID_Y_STEP)
    col = np.clip(col, 0, _hm_nx - 1)
    row = np.clip(row, 0, _hm_ny - 1)
    return col, row

def accumulate_heatmap(heatmap, X, Y, weight=1.0):
    col, row = world_to_heatmap_cell(X, Y)
    heatmap[row, col] += weight

def draw_trajectory(frame, tid, color):
    pts = list(track_history_px[tid])
    for i in range(1, len(pts)):
        alpha = i / len(pts)
        c = tuple(int(ch * alpha) for ch in color)
        cv2.line(frame, pts[i-1], pts[i], c, 2, cv2.LINE_AA)

def draw_extrapolated_path(frame, pos_w, vel_reg, horizon_s, steps=16):
    prev_px = world_to_pixel(pos_w[0], pos_w[1])
    for s in range(1, steps + 1):
        fw = pos_w + vel_reg * (horizon_s * s / steps)
        curr_px = world_to_pixel(fw[0], fw[1])
        if s % 2 == 1:
            cv2.line(frame, prev_px, curr_px, COLOR_TRAJ_PATH, 2, cv2.LINE_AA)
        prev_px = curr_px

def draw_hud(frame, alerts):
    if not alerts:
        return
    h = 32 + HUD_ROW_HEIGHT * len(alerts)
    w = 560
    ov = frame.copy()
    cv2.rectangle(ov, (6, 6), (w, h), (12, 12, 12), -1)
    cv2.addWeighted(ov, 0.65, frame, 0.35, 0, frame)
    cv2.rectangle(frame, (6, 6), (w, h), (70, 70, 70), 1)
    cv2.putText(frame, "COLLISION RISK MONITOR (KALMAN-STABILIZED)", (14, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.54, (220, 220, 220), 1, cv2.LINE_AA)
    
    order = {"CRITICAL": 0, "WARNING": 1, "MONITOR": 2}
    for i, (lvl, col, txt) in enumerate(sorted(alerts, key=lambda x: order[x[0]])):
        y_pos = 48 + HUD_ROW_HEIGHT * i
        # Color status indicator pill
        cv2.circle(frame, (18, y_pos - 5), 5, col, -1)
        cv2.putText(frame, txt, (28, y_pos), cv2.FONT_HERSHEY_SIMPLEX, HUD_FONT_SCALE, col, 2, cv2.LINE_AA)

collision_log = []
last_logged_frame = defaultdict(lambda: -999)

def fmt_timestamp(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    ms = int((seconds % 1) * 1000)
    return f"{h:02d}:{m:02d}:{s:02d}.{ms:03d}"

In [11]:
# Cell 9: Excel Exporter
def write_collision_xlsx(log, path):
    wb = Workbook()
    thin = Side(style="thin", color="AAAAAA")
    bdr = Border(left=thin, right=thin, top=thin, bottom=thin)
    hdr_fill = PatternFill("solid", start_color="1F3864")
    hdr_font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
    hdr_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    row_font = Font(name="Arial", size=9)
    ctr = Alignment(horizontal="center")
    fills = {
        "CRITICAL": PatternFill("solid", start_color="FFCCCC"),
        "WARNING":  PatternFill("solid", start_color="FFE5CC"),
        "MONITOR":  PatternFill("solid", start_color="FFFFCC"),
    }

    # Sheet 1: Full Event Log
    ws1 = wb.active
    ws1.title = "Collision Event Log"
    headers = [
        "Frame", "Timestamp", "Video Time (s)",
        "ID_1", "Class_1", "ID_2", "Class_2",
        "Scenario", "Risk Level",
        "TTC (s)", "DRAC (m/s²)",
        "MTTC (s) [rear-end]", "PET (s) [crossing]",
        "Distance (m)", "Rel Speed (m/s)",
    ]
    ws1.row_dimensions[1].height = 30
    for ci, h in enumerate(headers, 1):
        c = ws1.cell(row=1, column=ci, value=h)
        c.font = hdr_font; c.fill = hdr_fill
        c.alignment = hdr_align; c.border = bdr

    def _fmt(v): return round(v, 3) if v < 999 else "∞"
    for ri, ev in enumerate(log, 2):
        vals = [
            ev["frame"], ev["timestamp"], round(ev["time_s"], 3),
            ev["id1"], ev["cls1"], ev["id2"], ev["cls2"],
            ev["scenario"], ev["level"],
            round(ev["ttc"], 3), round(ev["drac"], 3),
            _fmt(ev["mttc"]), _fmt(ev["pet"]),
            round(ev["dist"], 2), round(ev["rel_speed"], 2),
        ]
        fill = fills.get(ev["level"], fills["MONITOR"])
        for ci, v in enumerate(vals, 1):
            c = ws1.cell(row=ri, column=ci, value=v)
            c.fill = fill; c.font = row_font
            c.alignment = ctr; c.border = bdr

    for i, w in enumerate([8,14,14,6,10,6,10,12,10,10,12,14,14,12,14], 1):
        ws1.column_dimensions[get_column_letter(i)].width = w
    ws1.freeze_panes = "A2"

    # Sheet 2: Pair Summary
    ws2 = wb.create_sheet("Pair Summary")
    s2h = [
        "ID_1","Class_1","ID_2","Class_2","Scenario",
        "# Frames","Min TTC (s)","Max DRAC (m/s²)",
        "Min MTTC (s)","Min PET (s)",
        "First Seen","Last Seen","Peak Level"
    ]
    for ci, h in enumerate(s2h, 1):
        c = ws2.cell(row=1, column=ci, value=h)
        c.font = hdr_font; c.fill = hdr_fill
        c.alignment = hdr_align; c.border = bdr
    ws2.row_dimensions[1].height = 30

    pairs = defaultdict(lambda: dict(count=0, ttcs=[], dracs=[], mttcs=[], pets=[],
                                     first="", last="", levels=[],
                                     cls1="", cls2="", scenario=""))
    for ev in log:
        k = (ev["id1"], ev["id2"])
        p = pairs[k]
        p["count"] += 1
        p["ttcs"].append(ev["ttc"])
        p["dracs"].append(ev["drac"])
        if ev["mttc"] < 999: p["mttcs"].append(ev["mttc"])
        if ev["pet"]  < 999: p["pets"].append(ev["pet"])
        if not p["first"]: p["first"] = ev["timestamp"]
        p["last"] = ev["timestamp"]
        p["levels"].append(ev["level"])
        p["cls1"] = ev["cls1"]
        p["cls2"] = ev["cls2"]
        p["scenario"] = ev["scenario"]

    order_map = {"CRITICAL": 0, "WARNING": 1, "MONITOR": 2}
    for ri, ((id1, id2), p) in enumerate(pairs.items(), 2):
        peak = min(p["levels"], key=lambda x: order_map[x])
        min_mttc = round(min(p["mttcs"]), 3) if p["mttcs"] else "∞"
        min_pet  = round(min(p["pets"]),  3) if p["pets"]  else "∞"
        vals = [
            id1, p["cls1"], id2, p["cls2"], p["scenario"],
            p["count"], round(min(p["ttcs"]), 3), round(max(p["dracs"]), 3),
            min_mttc, min_pet, p["first"], p["last"], peak
        ]
        fill = fills.get(peak, fills["MONITOR"])
        for ci, v in enumerate(vals, 1):
            c = ws2.cell(row=ri, column=ci, value=v)
            c.fill = fill; c.font = row_font
            c.alignment = ctr; c.border = bdr

    for i, w in enumerate([6,10,6,10,12,10,12,14,12,12,14,14,12], 1):
        ws2.column_dimensions[get_column_letter(i)].width = w
    ws2.freeze_panes = "A2"

    wb.save(path)
    print(f"✅ [XLSX] {len(log)} events, {len(pairs)} pairs saved to {path}")

In [12]:
# Cell 10: Video Slowdown Preprocessor (Optional)
def create_slow_video_if_requested(in_path, out_path, factor=0.5):
    """
    Slows video by factor (0.5 = 2x slower, doubles frames).
    Uses OpenCV VideoCapture/Writer if ffmpeg is not in PATH.
    """
    if pathlib.Path(out_path).exists():
        print(f"ℹ️ Slow video already exists: {out_path}")
        return out_path
    print(f"🎬 Generating slow-motion video ({factor}x speed) from {in_path}...")
    cap = cv2.VideoCapture(in_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, fps * factor, (w, h))
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        out.write(frame)
    cap.release()
    out.release()
    print(f"✅ Slowed video written to {out_path}")
    return out_path

if ENABLE_VIDEO_SLOWDOWN:
    EFFECTIVE_VIDEO_INPUT = create_slow_video_if_requested(INPUT_VIDEO, SLOW_VIDEO_PATH, SLOWDOWN_FACTOR)
else:
    EFFECTIVE_VIDEO_INPUT = INPUT_VIDEO

In [13]:
# Cell 11: Main Video Processing Loop (Kalman Filtering + Re-ID Smoothing + ByteTrack)
model = YOLO(MODEL_PATH)
reid_tracker = ReIDBridgeTracker(max_coast_frames=MAX_COAST_FRAMES, max_reid_dist=REID_MAX_DISTANCE_M)

cap = cv2.VideoCapture(EFFECTIVE_VIDEO_INPUT)
if not cap.isOpened():
    raise RuntimeError(f"❌ Failed to open video: {EFFECTIVE_VIDEO_INPUT}")

FPS = cap.get(cv2.CAP_PROP_FPS)
if FPS <= 1:
    FPS = 30.0
dt_nominal = 1.0 / FPS

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(VIDEO_OUT_PATH, fourcc, int(FPS), (960, 540))

prev_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
frame_idx = 0

# Metrics log for tracking accuracy calculation
per_frame_metrics_log = []

print(f"🚀 Processing video at {FPS:.1f} FPS with Kalman + Re-ID enhancement...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    curr_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
    dt = (curr_ms - prev_ms) / 1000.0
    if dt <= 0 or dt > 0.5:
        dt = dt_nominal
    prev_ms = curr_ms
    frame_idx += 1
    time_s = frame_idx / FPS

    frame = cv2.resize(frame, (960, 540))

    # 1. Overlay world coordinate grid
    draw_world_grid(frame)
    cv2.polylines(frame, [roi_pts], True, (70, 70, 70), 1)

    # 2. Tracking with Ultralytics ByteTrack
    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        conf=DETECTION_CONFIDENCE,
        verbose=False
    )[0]

    raw_detections = []
    frame_confs = []

    if results.boxes is not None and results.boxes.id is not None:
        boxes_xyxy = results.boxes.xyxy.cpu().numpy()
        track_ids  = results.boxes.id.int().cpu().numpy()
        classes    = results.boxes.cls.int().cpu().numpy()
        confs      = results.boxes.conf.cpu().numpy()

        for box, tid, cls, conf in zip(boxes_xyxy, track_ids, classes, confs):
            if cls not in [0, 1, 2]:
                continue
            x1, y1, x2, y2 = box
            u, v = (x1 + x2) / 2.0, y2  # Ground contact point

            # Filter by road ROI polygon with 10px buffer
            if cv2.pointPolygonTest(roi_pts, (float(u), float(v)), True) < -10.0:
                continue

            X, Y = pixel_to_world(u, v)
            raw_detections.append({
                'id': int(tid),
                'cls': int(cls),
                'world_pos': np.array([X, Y]),
                'bbox': (int(x1), int(y1), int(x2), int(y2)),
                'px': (int(u), int(v)),
                'conf': float(conf)
            })
            frame_confs.append(float(conf))

    # 3. Step Re-ID Bridge & Kalman Filter on World Coordinates
    active_tracks = reid_tracker.step(raw_detections, dt)

    vehicles = {}
    current_active_ids = set()

    for tr in active_tracks:
        tid = tr['id']
        cls = tr['cls']
        X, Y = tr['world_pos']
        u, v = tr['px']
        x1, y1, x2, y2 = tr['bbox']

        current_active_ids.add(tid)
        track_id_to_class[tid] = cls

        hw = track_history_world[tid]
        # Outlier rejection for extreme homography boundary jumps
        if len(hw) > 0 and np.linalg.norm(np.array([X, Y]) - hw[-1]) > 10.0:
            continue

        hw.append(np.array([X, Y]))
        track_history_px[tid].append((int(u), int(v)))

        vel_ema = compute_ema_velocity(tid, dt)
        vel_reg = fit_regression_velocity(tid)
        acc     = compute_acceleration(tid, vel_ema, dt)

        raw_sp = np.linalg.norm(vel_ema) * 3.6
        a_sp   = adaptive_alpha(ALPHA_SPEED, dt)
        speed  = a_sp * raw_sp + (1.0 - a_sp) * track_speed[tid]
        track_speed[tid] = speed

        color = class_colors.get(cls, (0, 255, 0))
        label = class_names.get(cls, "Vehicle")

        vehicles[tid] = {
            "pos": np.array([X, Y]),
            "vel_ema": vel_ema,
            "vel_reg": vel_reg,
            "acc": acc,
            "px": (int(u), int(v)),
            "cls": cls,
            "speed": speed,
            "color": color,
        }

        # High-visibility vehicle HUD & bounding box
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        tag = f"{label} ID:{tid} {speed:.1f}km/h"
        (tw, th), _ = cv2.getTextSize(tag, cv2.FONT_HERSHEY_SIMPLEX, VEHICLE_LABEL_SCALE, 1)
        tag_y = max(th + 4, y1 - 6)
        cv2.rectangle(frame, (x1, tag_y - th - 3), (x1 + tw + 4, tag_y + 3), (20, 20, 20), -1)
        cv2.putText(frame, tag, (x1 + 2, tag_y), cv2.FONT_HERSHEY_SIMPLEX, VEHICLE_LABEL_SCALE, color, 1, cv2.LINE_AA)
        draw_trajectory(frame, tid, color)
        draw_extrapolated_path(frame, np.array([X, Y]), vel_reg, TTC_MAX_S)

    # 4. Periodic stale track cleanup
    if frame_idx % CLEANUP_EVERY == 0:
        for store in (track_history_world, track_history_px, velocity_ema,
                      accel_ema, prev_vel_for_acc, track_speed, track_id_to_class):
            for dead_id in list(store.keys()):
                if dead_id not in current_active_ids:
                    del store[dead_id]

    # 5. Pairwise Conflict Evaluation with Refined Persistence
    current_pairs_at_risk = set()

    for (id1, veh1), (id2, veh2) in itertools.combinations(vehicles.items(), 2):
        pair_key = (min(id1, id2), max(id1, id2))

        p1, v1r = veh1["pos"], veh1["vel_reg"]
        p2, v2r = veh2["pos"], veh2["vel_reg"]
        v1e, a1 = veh1["vel_ema"], veh1["acc"]
        v2e, a2 = veh2["vel_ema"], veh2["acc"]

        # Distance gate
        dist_now = np.linalg.norm(p2 - p1)
        if dist_now > DISTANCE_GATE_M:
            pair_risk_window[pair_key].append(0)
            continue

        # Closest approach
        t_ca, d_ca, p_ca = closest_approach(p1, v1r, p2, v2r)
        if t_ca <= 0 or t_ca > TTC_MAX_S or d_ca > MIN_APPROACH_DIST_M:
            pair_risk_window[pair_key].append(0)
            continue

        lat_sep = lateral_separation(p1, v1r, p2)
        following = is_following_scenario(v1r, v2r, p1, p2)

        if not following and lat_sep > SIDE_BY_SIDE_LATERAL_M:
            pair_risk_window[pair_key].append(0)
            continue

        if following:
            avg_dir = v1r + v2r
            anorm = np.linalg.norm(avg_dir)
            if anorm > 1e-3:
                avg_dir /= anorm
                if abs(np.dot(p2 - p1, avg_dir)) > FOLLOWING_LONGIT_GAP_M * 3:
                    pair_risk_window[pair_key].append(0)
                    continue

        DRAC_val = compute_drac(p1, p2, v1e, v2e)
        scenario = "REAR-END" if following else "CROSSING"

        if following:
            MTTC_val = compute_mttc(p1, v1e, a1, p2, v2e, a2)
            PET_val  = np.inf
            level, alert_color = risk_tier_rear_end(t_ca, DRAC_val, MTTC_val)
            metric_str = f"TTC:{t_ca:.2f}s  DRAC:{DRAC_val:.2f}  MTTC:{'∞' if MTTC_val>999 else f'{MTTC_val:.2f}'}s"
        else:
            PET_val  = compute_pet(p1, v1r, p2, v2r, p_ca)
            MTTC_val = np.inf
            level, alert_color = risk_tier_crossing(PET_val)
            metric_str = f"TTC:{t_ca:.2f}s  PET:{'∞' if PET_val>999 else f'{PET_val:.2f}'}s  DRAC:{DRAC_val:.2f}"

        # Check if condition qualifies as at-risk
        is_risky = (DRAC_val >= DRAC_WARNING) or (t_ca < MTTC_WARNING_S) or (PET_val < PET_WARNING_S)
        pair_risk_window[pair_key].append(1 if is_risky else 0)

        # REFINED PERSISTENCE: Alert triggers only if >= RISK_PERSIST_FRAMES within RISK_WINDOW_SIZE
        # This protects against 1-frame drops while preventing single-frame transient spikes!
        if sum(pair_risk_window[pair_key]) >= RISK_PERSIST_FRAMES:
            current_pairs_at_risk.add(pair_key)

            # Heatmap Accumulation
            if following and MTTC_val < 1.5:
                accumulate_heatmap(heatmap_mttc, p_ca[0], p_ca[1], weight=1.0)
            elif (not following) and PET_val < 2.0:
                accumulate_heatmap(heatmap_pet, p_ca[0], p_ca[1], weight=1.0)

            stable_alerts[pair_key] = {
                "level": level, "color": alert_color,
                "TTC": t_ca, "DRAC": DRAC_val,
                "MTTC": MTTC_val, "PET": PET_val,
                "p_ca": p_ca, "id1": id1, "id2": id2,
                "scenario": scenario, "metric_str": metric_str,
            }

            if frame_idx - last_logged_frame[pair_key] >= LOG_INTERVAL_FRAMES:
                last_logged_frame[pair_key] = frame_idx
                collision_log.append({
                    "frame": frame_idx, "timestamp": fmt_timestamp(time_s),
                    "time_s": time_s, "id1": id1, "cls1": class_names.get(veh1["cls"], "Vehicle"),
                    "id2": id2, "cls2": class_names.get(veh2["cls"], "Vehicle"),
                    "scenario": scenario, "level": level,
                    "ttc": t_ca, "drac": DRAC_val,
                    "mttc": MTTC_val, "pet": PET_val,
                    "dist": dist_now, "rel_speed": float(np.linalg.norm(v2e - v1e)),
                })

    # Remove stale alerts
    for pk in list(stable_alerts.keys()):
        if pk not in current_pairs_at_risk:
            del stable_alerts[pk]

    # Draw active alerts
    hud_rows = []
    for pk, al in stable_alerts.items():
        id1, id2 = al["id1"], al["id2"]
        alert_color = al["color"]
        cpx = world_to_pixel(al["p_ca"][0], al["p_ca"][1])

        cv2.circle(frame, cpx, 9, alert_color, -1)
        cv2.circle(frame, cpx, 14, alert_color, 2)

        if id1 in vehicles and id2 in vehicles:
            cv2.line(frame, vehicles[id1]["px"], vehicles[id2]["px"], alert_color, 2, cv2.LINE_AA)

        hud_rows.append((
            al["level"], alert_color,
            f"[{al['level']}|{al['scenario']}] ID{id1}↔ID{id2}  {al['metric_str']}"
        ))

    draw_hud(frame, hud_rows)

    # Footer timestamp & statistics
    cv2.putText(
        frame,
        f"Frame:{frame_idx}  {fmt_timestamp(time_s)}  Tracks:{len(vehicles)}",
        (600, 537), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (140, 140, 140), 1, cv2.LINE_AA
    )
    out.write(frame)

    per_frame_metrics_log.append({
        'frame': frame_idx,
        'num_tracks': len(vehicles),
        'confs': frame_confs,
        'velocities': [v['vel_ema'] for v in vehicles.values()],
        'track_ids': list(vehicles.keys())
    })

cap.release()
out.release()
print(f"🎬 Video processing complete! Rendered to {VIDEO_OUT_PATH}")

🚀 Processing video at 100.0 FPS with Kalman + Re-ID enhancement...
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 2 packages in 329ms
Prepared 1 package in 77ms
Installed 1 package in 2ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

🎬 Video processing complete! Rendered to final_output_tracked.mp4


In [14]:
# Cell 13: Quantitative Accuracy & Tracking Stability Metrics
# ================================================================
# Evaluates pipeline quality without ground-truth bounding box labels
# ================================================================
print("=" * 60)
print("📊 QUANTITATIVE PIPELINE ACCURACY & STABILITY REPORT")
print("=" * 60)

# 1. Mean Detection Confidence
all_confs = [c for f in per_frame_metrics_log for c in f['confs']]
mean_conf = np.mean(all_confs) if all_confs else 0.0

# 2. Track Continuity & ID Longevity
all_seen_ids = set()
id_frame_counts = defaultdict(int)
for f in per_frame_metrics_log:
    for tid in f['track_ids']:
        all_seen_ids.add(tid)
        id_frame_counts[tid] += 1

total_unique_tracks = len(all_seen_ids)
avg_track_duration  = np.mean(list(id_frame_counts.values())) if id_frame_counts else 0
# Fragmented tracks (lasting fewer than 5 frames = likely noise/flicker)
short_lived_tracks  = sum(1 for c in id_frame_counts.values() if c < 5)
fragmentation_rate  = (short_lived_tracks / max(1, total_unique_tracks)) * 100.0

# 3. Trajectory Angular Smoothness (Heading Variation)
angular_diffs = []
for tid, h in track_history_world.items():
    if len(h) < 3:
        continue
    arr = np.array(h)
    diffs = np.diff(arr, axis=0)
    norms = np.linalg.norm(diffs, axis=1, keepdims=True)
    valid = (norms[:, 0] > 0.05)
    if np.sum(valid) < 2:
        continue
    unit_dirs = diffs[valid] / norms[valid]
    for i in range(1, len(unit_dirs)):
        dot = np.clip(np.dot(unit_dirs[i-1], unit_dirs[i]), -1.0, 1.0)
        theta = np.arccos(dot)  # radians
        angular_diffs.append(theta)

mean_angular_jitter_rad = np.mean(angular_diffs) if angular_diffs else 0.0
mean_angular_jitter_deg = np.degrees(mean_angular_jitter_rad)

# 4. Composite Tracking Stability Index (TSI, 0 to 100%)
# High confidence + low fragmentation + smooth headings = High TSI
conf_score   = np.clip(mean_conf * 100, 0, 100)
frag_penalty = np.clip(fragmentation_rate * 0.8, 0, 40)
jitter_pen   = np.clip((mean_angular_jitter_deg / 45.0) * 30, 0, 30)
tsi_score    = np.clip(conf_score - frag_penalty - jitter_pen, 0, 100)

print(f"1. Mean YOLO Detection Confidence : {mean_conf:.3f} ({conf_score:.1f}%)")
print(f"2. Total Unique Tracked Vehicles   : {total_unique_tracks}")
print(f"3. Average Track Length (frames)   : {avg_track_duration:.1f} frames ({avg_track_duration/FPS:.2f} s)")
print(f"4. Track Fragmentation Rate        : {fragmentation_rate:.2f}% (short-lived tracks < 5 frames)")
print(f"5. Trajectory Heading Jitter       : {mean_angular_jitter_deg:.2f}° mean angular variance")
print("-" * 60)
print(f"⭐ COMPOSITE TRACKING STABILITY INDEX : {tsi_score:.1f} / 100")
print("=" * 60)

📊 QUANTITATIVE PIPELINE ACCURACY & STABILITY REPORT
1. Mean YOLO Detection Confidence : 0.719 (71.9%)
2. Total Unique Tracked Vehicles   : 297
3. Average Track Length (frames)   : 46.7 frames (0.47 s)
4. Track Fragmentation Rate        : 28.62% (short-lived tracks < 5 frames)
5. Trajectory Heading Jitter       : 0.57° mean angular variance
------------------------------------------------------------
⭐ COMPOSITE TRACKING STABILITY INDEX : 48.6 / 100
